<a href="https://colab.research.google.com/github/nicolasramirezperilla/DataWave-Project/blob/master/Consolidado_Balance_Fiduciaria.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#1) Instalar librerias y conexión al servidor.






In [ ]:
from google.colab import auth
import pandas as pd
import numpy as np
from datetime import datetime
import gspread
from google.auth import default
import re
from googleapiclient.discovery import build

# Autenticación para Google Sheets
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

#2) Descargar información & definir parametros.



In [ ]:
# Configuración inicial
input_workbook_name = 'Consolidado Balances - Fiduciaria'
spreadsheet = gc.open(input_workbook_name)

# Obtener datos de las hojas
sheet1 = spreadsheet.worksheet("Balance_Resultado")
sheet2 = spreadsheet.worksheet("Parametria_1")
sheet4 = spreadsheet.worksheet("Inputs")
sheet5 = spreadsheet.worksheet("Parametria_2")

# Obtener fecha a actualizar y fecha anterior
fecha_actualizar = sheet4.acell('B1').value
fecha_anterior = sheet4.acell('B2').value

# Definir rango y obtener nombres de archivos desde enlaces
spreadsheet_id = '1t1EDsT9HQQkYL5QnsmV0u2wDagduXrFm4OHqjqak5-8'
worksheet = gc.open_by_key(spreadsheet_id).worksheet('Inputs')
start_row, end_row = 5, 100

for row in range(start_row, end_row + 1):
    cell_value = worksheet.acell(f'B{row}').value
    if cell_value:
        match = re.search(r'[-\w]{25,}', cell_value)
        if match:
            file_id = match.group(0)
            try:
                drive_service = build('drive', 'v3', credentials=creds)
                file_name = drive_service.files().get(fileId=file_id).execute()['name']
                worksheet.update_acell(f'C{row}', file_name)
            except Exception:
                continue

# Función para buscar el valor en la matriz basada en la fecha a actualizar
def buscar_valor(matriz, fecha):
    for fila in matriz:
        if len(fila) >= 3 and fila[0] == fecha:
            return fila[2]
    return None

# Obtener matriz y buscar el nombre del archivo a abrir
matriz = sheet4.get('A4:C')
input_workbook_name2 = buscar_valor(matriz, fecha_actualizar)
spreadsheet2 = gc.open(input_workbook_name2)

# Construcción de DataFrames a partir de los datos obtenidos
df_temp = pd.DataFrame(sheet1.get_all_values()[1:], columns=sheet1.get_all_values()[0])
indice_inicio = df_temp.columns.get_loc('cuenta')
indice_final = df_temp.columns.get_loc(fecha_anterior)
df1 = df_temp.iloc[:, indice_inicio:indice_final + 1]

df2 = pd.DataFrame(sheet2.get_all_values()[1:], columns=sheet2.get_all_values()[0])
df2['cuenta'] = df2['cuenta'].str.replace(',', '').astype(int)

df22 = pd.DataFrame(sheet5.get_all_values()[1:], columns=sheet5.get_all_values()[0])
df22['cuenta'] = df22['cuenta'].str.replace(',', '').astype(int)

sheet3 = spreadsheet2.worksheet(input_workbook_name2)
df4 = pd.DataFrame(sheet3.get_all_values()[1:], columns=sheet3.get_all_values()[0])
df4 = df4.rename(columns={'cuenta': 'cuenta 1', 'nombre_cuenta': 'nombre_cuenta 1'})

df4['saldo_final_export'] = df4['saldo_final_export'].str.replace('.', '', regex=False)
df4['saldo_final_export'] = df4['saldo_final_export'].str.replace(',', '.')
df4['saldo_final_export'] = pd.to_numeric(df4['saldo_final_export'])
df4['saldo_final_export'] = df4.apply(lambda x: x['saldo_final_export'] * -1 if x['naturaleza'] == 'C' else x['saldo_final_export'], axis=1)

# 3) Cruce bases.

Cruces:
*   Parameterias
*   Balance Primario: Historico / Balance Mensual
*   Balance Secundario: Balance Primario / Parametrias

In [ ]:
# Realizar el primer merge
merged_df1 = df2.rename(columns={'cuenta': 'cuenta 1', 'nombre_cuenta': 'nombre_cuenta 1'})
merged_df1['cuenta 1'] = merged_df1['cuenta 1'].astype(int)
df1['cuenta'] = df1['cuenta'].astype(str).str.replace(',', '').astype(int)
merged_df2 = pd.merge(merged_df1, df1, left_on="cuenta 1", right_on="cuenta", how="outer").drop(columns=['cuenta 1', 'nombre_cuenta 1'], errors='ignore')

#df4['cuenta 1'] = df4['cuenta 1'].str.replace(',', '').astype(int)
df4['cuenta 1'] = df4['cuenta 1'].astype(int)

# Realizar el segundo merge con df4
merged_df3 = pd.merge(merged_df2, df4[['cuenta 1', 'nombre_cuenta 1', 'saldo_final_export']], left_on="cuenta", right_on="cuenta 1", how="left").drop(columns=['cuenta 1', 'nombre_cuenta 1'])
merged_df3 = merged_df3.rename(columns={'saldo_final_export': fecha_actualizar})
merged_df3['cuenta'] = merged_df3['cuenta'].astype(int)


<ipython-input-3-5fa53e097199>:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1['cuenta'] = df1['cuenta'].astype(str).str.replace(',', '').astype(int)


#4) Base de Nuevas Cuentas + Concantenado Balance

In [ ]:
merged_df44 = pd.merge(df4[['cuenta 1', 'nombre_cuenta 1', 'saldo_final_export']], merged_df2[['cuenta']], left_on="cuenta 1", right_on="cuenta", how='left', indicator=True)
merged_df44 = merged_df44[merged_df44['_merge'] == 'left_only'].drop(columns=['_merge'])
merged_df44 = merged_df44.drop_duplicates(subset=['cuenta 1', 'nombre_cuenta 1'], keep=False)
merged_df44 = merged_df44.drop(columns='cuenta', errors='ignore')
merged_df44 = merged_df44.rename(columns={'cuenta 1': 'cuenta','nombre_cuenta 1': 'nombre_cuenta','saldo_final_export':'valor'})
merged_df44['cuenta'] = merged_df44['cuenta'].fillna('0')
#merged_df44['cuenta'] = merged_df44['cuenta'].str.replace(',', '').astype(int)
merged_df44['Parameteria'] = merged_df44['cuenta'].apply(lambda x: any(str(x) in str(y) for y in df2['cuenta']))
merged_df44['Min_Lvl'] = merged_df44['cuenta'].apply(lambda x: not any(merged_df44['cuenta'].astype(str).str.startswith(str(x)) & (merged_df44['cuenta'] != x)))

merged_df4 = merged_df44.rename(columns={'valor': fecha_actualizar})

# Concatenar DataFrames
merged_df_concat = pd.concat([merged_df3, merged_df4], ignore_index=True)
merged_df_concat = merged_df_concat.drop(columns=['Parameteria', 'Min_Lvl'])
merged_df_concat = merged_df_concat.drop_duplicates(subset=['cuenta', 'nombre_cuenta'])

#5) Formato + Clasificaciones

1.   Formato de número
2.   Clasificacion tipo (B,P&L,O)
3.   Renombramiento columnas
4.   Reordenamiento columnas
6.   Creación filas "detalle"
7.   Clasificacion codigo (001,002,003)
7.   Clasificacion minimo lvl (TRUE,FALSE)
9.   Interpretación valores NaN





In [ ]:
# 1. Definir funciones
# ---------------------

# Función para convertir cadenas numéricas a formato numérico
def convert_to_numeric(value):
    try:
        if isinstance(value, str):
            return float(value.replace('.', '').replace(',', '.'))
        return float(value)
    except ValueError:
        return None

def formato_solo_enteros(valor):
    try:
        valor = int(valor)  # Intentar convertir a entero
        return valor
    except (TypeError, ValueError):
        if isinstance(valor, str) and '.' in valor:
            # Si es una cadena y contiene una coma, eliminar todo después de la coma
            valor = valor.split('.')[0]
        return valor

# Función para categorizar cuentas
def categorizar_cuenta(cuenta):
    cuenta_str = str(cuenta) if pd.notna(cuenta) else ''
    if cuenta_str.startswith(('1', '2', '3')):
        return 'B'
    elif cuenta_str.startswith(('4', '5')):
        return 'P&L'
    else:
        return 'O'

# Función para asignar códigos según la cuenta
def assign_code(row):
    nombre_cuenta_str = str(row['nombre_cuenta'])
    cuenta_str = str(row['cuenta'])
    if 'detalle' in nombre_cuenta_str:
        return '003'
    elif cuenta_str.startswith(('1', '2', '3')):
        return '001'
    elif cuenta_str.startswith(('4', '5')):
        return '002'
    else:
        return ''

# Función para eliminar nombres de cuenta con más de un "+ detalle"
def remove_extra_detalle(name):
    if name.count(' + detalle') > 1:
        return None
    return name

# 2. Preparar y limpiar datos
# ----------------------------

# Convertir las columnas de fechas a numéricas
fechas = [col for col in merged_df_concat.columns if '/' in col]
for fecha in fechas:
    merged_df_concat[fecha] = merged_df_concat[fecha].astype(str)
    merged_df_concat[fecha] = merged_df_concat[fecha].str.replace(',', '', regex=False)
    merged_df_concat[fecha] = pd.to_numeric(merged_df_concat[fecha], errors='coerce')

# Filtrar cuentas que comienzan con 4 o 5
df_filtered = merged_df_concat[merged_df_concat['cuenta'].astype(str).str.startswith(('4', '5'))]

# 3. Crear nuevas filas con "+ detalle"
# --------------------------------------

new_rows = []

for _, row in df_filtered.iterrows():
    cuenta = row['cuenta']
    nombre_cuenta = row['nombre_cuenta']
    detalle_name = f"{nombre_cuenta} + detalle"

    existing_detalle_row = merged_df_concat[merged_df_concat['nombre_cuenta'] == detalle_name]

    if existing_detalle_row.empty:
        new_row = {'cuenta': cuenta, 'nombre_cuenta': detalle_name}

        prev_value = row[fechas[0]]
        differences = [0]  # La diferencia para el primer mes es 0

        for fecha in fechas[1:]:
            current_value = row[fecha]
            difference = current_value - prev_value
            differences.append(difference)
            prev_value = current_value

        new_row.update(dict(zip(fechas, differences)))
        new_rows.append(new_row)

    else:
        existing_row_index = existing_detalle_row.index[0]
        prev_value = row[fechas[0]]

        updated_differences = [0]

        for fecha in fechas[1:]:
            current_value = row[fecha]
            difference = current_value - prev_value
            updated_differences.append(difference)
            prev_value = current_value

        merged_df_concat.loc[existing_row_index, fechas] = updated_differences

# Convertir las nuevas filas a un DataFrame y concatenar con el original
new_df = pd.DataFrame(new_rows)
merged_df_concat = pd.concat([merged_df_concat, new_df], ignore_index=True)

# 4. Limpiar y ajustar datos
# ---------------------------

# Eliminar filas con más de un "+ detalle" y limpiar columnas
merged_df_concat['nombre_cuenta'] = merged_df_concat['nombre_cuenta'].apply(remove_extra_detalle)
merged_df_concat = merged_df_concat.dropna(subset=['nombre_cuenta'])
#merged_df_concat = merged_df_concat.replace('nan', '', regex=True)
merged_df_concat['cuenta'] = merged_df_concat['cuenta'].astype(str)

# Aplicar categorización de cuentas
merged_df_concat['TIPO'] = merged_df_concat['cuenta'].apply(categorizar_cuenta)
columnas = ['TIPO'] + [col for col in merged_df_concat if col != 'TIPO']
merged_df_concat = merged_df_concat[columnas]

# Renombrar columnas del DataFrame
original_columns = ['TIPO', 'LOCAL', 'LOCAL II', 'COD CONSOLIDACION','NOMBRE CONSO', 'COD DE GESTION', 'NOMBRE GESTION']
new_columns = ['TIPO', 'CM', 'CM Detalle', 'Neocon', 'Neocon Detalle', 'Gestión', 'Gestión Detalle']
column_mapping = dict(zip(original_columns, new_columns))
merged_df_concat = merged_df_concat.rename(columns=column_mapping)

# Convertir columnas a formato numérico
merged_df_concat[fecha_actualizar] = merged_df_concat[fecha_actualizar].apply(convert_to_numeric)
columns_to_fill = merged_df_concat.columns[merged_df_concat.columns.get_loc('nombre_cuenta') + 1:]
merged_df_concat[columns_to_fill] = merged_df_concat[columns_to_fill].fillna(value=0)
merged_df_concat[columns_to_fill] = merged_df_concat[columns_to_fill].applymap(formato_solo_enteros)


# 5. Asignar valores y reorganizar columnas
# ------------------------------------------

merged_df_concat_new = merged_df_concat.copy()

# Crear la nueva columna 'Clasificacion'
merged_df_concat_new['Clasificacion'] = merged_df_concat_new.apply(assign_code, axis=1)

# Crear la nueva columna 'Min_Lvl'
merged_df_concat_new['Min_Lvl'] = merged_df_concat['cuenta'].apply(lambda x: not any(merged_df_concat['cuenta'].str.startswith(x) & (merged_df_concat['cuenta'] != x)))

# Reordenar las columnas para colocar 'Clasificacion' y 'Min_Lvl' antes de 'cuenta'
columns = merged_df_concat_new.columns.tolist()
cuenta_index = columns.index('cuenta')

columns.insert(cuenta_index, columns.pop(columns.index('Clasificacion')))
columns.insert(cuenta_index, columns.pop(columns.index('Min_Lvl')))
merged_df_concat_new = merged_df_concat_new[columns]

# 6. Finalizar el DataFrame
# -------------------------

# Convertir la columna 'cuenta' a formato numérico
merged_df_concat_new = merged_df_concat_new[merged_df_concat_new['cuenta'].str.strip() != '']
merged_df_concat_new['cuenta'] = merged_df_concat_new['cuenta'].str.replace(',', '').astype(float)

# Ordenar el DataFrame por la columna 'cuenta'
merged_df_concat_new = merged_df_concat_new.sort_values(by='cuenta', ascending=True)

<ipython-input-5-b860d2606636>:134: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  merged_df_concat[columns_to_fill] = merged_df_concat[columns_to_fill].applymap(formato_solo_enteros)


#6)

In [ ]:
merged_df_concat_new_1 = merged_df_concat.drop_duplicates(subset=['cuenta'])
merged_df_concat_new_1 = merged_df_concat_new.fillna("")
merged_df_concat_new_1 = merged_df_concat_new_1.loc[~(merged_df_concat_new_1[['CM']].eq("")).any(axis=1)]

cuenta_pos = merged_df_concat_new_1.columns.get_loc('cuenta')
columns_to_keep = ['CM'] + list(merged_df_concat_new_1.columns[cuenta_pos+1:])
merged_df_concat_new_2 = merged_df_concat_new_1[columns_to_keep]

PyL_2CR_acumulado = merged_df_concat_new_2[~merged_df_concat_new_2['nombre_cuenta'].str.contains('\+ detalle', na=False, regex=True)]
PyL_2CR_acumulado_agrupado = PyL_2CR_acumulado.groupby('CM').sum().reset_index().drop(columns=['nombre_cuenta'])

date_columns = PyL_2CR_acumulado_agrupado.columns[1:]  # Ajusta el índice si es necesario
def calculate_differences(group):
    for i in range(len(date_columns) - 1):
        current_col = date_columns[i]
        next_col = date_columns[i + 1]
        group[f'{current_col} - {next_col}'] = group[next_col] - group[current_col]
    return group

PyL_2CR_estanco = PyL_2CR_acumulado_agrupado.groupby('CM').apply(calculate_differences)
PyL_2CR_estanco = PyL_2CR_estanco.reset_index(drop=True).rename(columns={'CM': 'LOCAL'})
# Número de filas existentes en el DataFrame
# Número de nuevas filas a agregar (por ejemplo, 5)
num_filas_a_agregar = 17

# Crear un nuevo DataFrame con las nuevas filas
nuevas_filas = pd.DataFrame({
    'LOCAL': [f'c{i+1}' for i in range(num_filas_a_agregar)],
    'CONCEPTOS': [""] * num_filas_a_agregar  # Rellena con 0 u otro valor si lo necesitas
})
PyL_2CR_estanco = pd.concat([PyL_2CR_estanco, nuevas_filas], ignore_index=True)

dv_to_description = {
    'MII': 'Ingresos Financieros',
    'MIC': 'Costes Financieros',
    'MI': 'MARGEN DE INTERES',
    'c1': 'Comisiones Netas',
    'c2': 'Comisiones Recibidas',
    'CFAM': 'Admon. de Fondos',
    'CFID': 'De Fiducia',
    'c3': 'Comisiones Pagadas',
    'CRED': 'Admon. de Fondos',
    'OC': 'Otras',
    'c4': 'Operaciones Financieras ROF',
    'DC': 'Diferencia de Cambio',
    'OF': 'Resto Operaciones Financieras',
    'c5': 'Resto de Ingresos Netos Ordinarios',
    'DV': 'Rendimiento de Instrumentos de Capital (Divid.)',
    'c6': 'Resultados Puesta en Equivalencia',
    'c7': 'Rto. Otros Productos y Cargas',
    'RNO': 'Otros Extraordinarios',
    'c8': 'MARGEN BRUTO',
    'c9': 'TOTAL GASTO + AMORTIZACIONES',
    'c10': 'Gtos Gen. Administración',
    'PER': 'Gtos. Personal',
    'GG': 'Gastos Generales',
    'IMP': 'Tributos',
    'DYA': 'Amortizaciones',
    'c11': 'MARGEN NETO',
    'c12': 'Perdida por Deterioro de Activos',
    'OI': 'Dotación Insolvencias neta de Recuperaciones',
    'c13': 'Pérdida de Deterioro Resto de Activo',
    'DP': 'Dotaciones a Provisiones',
    'c14': 'RDOS DE EXPLOTACIÓN',
    'c15': 'Resto de Resultados No Ordinarios',
    'c16': 'BAI',
    'RTA': 'Impuesto Sociedades',
    'c17': 'Bº NETO'
}

PyL_2CR_estanco['CONCEPTOS'] = PyL_2CR_estanco['LOCAL'].map(dv_to_description)

columns = list(PyL_2CR_estanco.columns)
dv_index = columns.index('LOCAL')
new_order = columns[:dv_index + 1] + ['CONCEPTOS'] + columns[dv_index + 1:-1]
PyL_2CR_estanco = PyL_2CR_estanco[new_order]

# Obtén el orden de las filas según el diccionario
order = list(dv_to_description.keys())
PyL_2CR_estanco = PyL_2CR_estanco[PyL_2CR_estanco['LOCAL'].isin(order)]
PyL_2CR_estanco = PyL_2CR_estanco.set_index('LOCAL').reindex(order).reset_index()
PyL_2CR_estanco = PyL_2CR_estanco.fillna(0)


MI_sum = PyL_2CR_estanco.loc[PyL_2CR_estanco['LOCAL'].isin(['MII', 'MIC']), PyL_2CR_estanco.columns[2:]].fillna(0).sum()
c2_sum = PyL_2CR_estanco.loc[PyL_2CR_estanco['LOCAL'].isin(['CFAM', 'CFID']), PyL_2CR_estanco.columns[2:]].fillna(0).sum()
c3_sum = PyL_2CR_estanco.loc[PyL_2CR_estanco['LOCAL'].isin(['CRED', 'OC']), PyL_2CR_estanco.columns[2:]].fillna(0).sum()
c1_sum = PyL_2CR_estanco.loc[PyL_2CR_estanco['LOCAL'].isin(['CFAM', 'CFAM','CRED','OC']), PyL_2CR_estanco.columns[2:]].fillna(0).sum()
c4_sum = PyL_2CR_estanco.loc[PyL_2CR_estanco['LOCAL'].isin(['MII', 'MIC']), PyL_2CR_estanco.columns[2:]].fillna(0).sum()
c5_sum = PyL_2CR_estanco.loc[PyL_2CR_estanco['LOCAL'].isin(['DC', 'OF']), PyL_2CR_estanco.columns[2:]].fillna(0).sum()
c6_sum = PyL_2CR_estanco.loc[PyL_2CR_estanco['LOCAL'].isin(['c6']), PyL_2CR_estanco.columns[2:]].fillna(0).sum()
c7_sum = PyL_2CR_estanco.loc[PyL_2CR_estanco['LOCAL'].isin(['RNO']), PyL_2CR_estanco.columns[2:]].fillna(0).sum()
c8_sum = PyL_2CR_estanco.loc[PyL_2CR_estanco['LOCAL'].isin(['MI', 'CFAM', 'CFAM','CRED','OC','MII', 'MIC','DC', 'OF']), PyL_2CR_estanco.columns[2:]].fillna(0).sum()
c9_sum = PyL_2CR_estanco.loc[PyL_2CR_estanco['LOCAL'].isin(['PER', 'GG','IMP', 'DYA']), PyL_2CR_estanco.columns[2:]].fillna(0).sum()
c10_sum = PyL_2CR_estanco.loc[PyL_2CR_estanco['LOCAL'].isin(['PER', 'GG','IMP']), PyL_2CR_estanco.columns[2:]].fillna(0).sum()
c11_sum = PyL_2CR_estanco.loc[PyL_2CR_estanco['LOCAL'].isin(['MI', 'CFAM', 'CFAM','CRED','OC','MII', 'MIC','DC', 'OF', 'PER', 'GG','IMP', 'DYA']), PyL_2CR_estanco.columns[2:]].fillna(0).sum()
c12_sum = PyL_2CR_estanco.loc[PyL_2CR_estanco['LOCAL'].isin(['OI', 'DP']), PyL_2CR_estanco.columns[2:]].fillna(0).sum()
c13_sum  = PyL_2CR_estanco.loc[PyL_2CR_estanco['LOCAL'].isin(['c13']), PyL_2CR_estanco.columns[2:]].fillna(0).sum()
c14_sum = PyL_2CR_estanco.loc[PyL_2CR_estanco['LOCAL'].isin(['MI', 'CFAM', 'CFAM','CRED','OC','MII', 'MIC','DC', 'OF', 'PER', 'GG','IMP', 'DYA', 'OI', 'DP']), PyL_2CR_estanco.columns[2:]].fillna(0).sum()
c15_sum = PyL_2CR_estanco.loc[PyL_2CR_estanco['LOCAL'].isin(['c15']), PyL_2CR_estanco.columns[2:]].fillna(0).sum()
c16_sum = PyL_2CR_estanco.loc[PyL_2CR_estanco['LOCAL'].isin(['MI', 'CFAM', 'CFAM','CRED','OC','MII', 'MIC','DC', 'OF', 'PER', 'GG','IMP', 'DYA', 'OI', 'DP', 'c15']), PyL_2CR_estanco.columns[2:]].fillna(0).sum()
c17_sum = PyL_2CR_estanco.loc[PyL_2CR_estanco['LOCAL'].isin(['MI', 'CFAM', 'CFAM','CRED','OC','MII', 'MIC','DC', 'OF', 'PER', 'GG','IMP', 'DYA', 'OI', 'DP', 'c15', 'RTA']), PyL_2CR_estanco.columns[2:]].fillna(0).sum()

PyL_2CR_estanco.loc[PyL_2CR_estanco['LOCAL'] == 'MI', PyL_2CR_estanco.columns[2:]] = MI_sum.values
PyL_2CR_estanco.loc[PyL_2CR_estanco['LOCAL'] == 'c1', PyL_2CR_estanco.columns[2:]] = c1_sum.values
PyL_2CR_estanco.loc[PyL_2CR_estanco['LOCAL'] == 'c2', PyL_2CR_estanco.columns[2:]] = c2_sum.values
PyL_2CR_estanco.loc[PyL_2CR_estanco['LOCAL'] == 'c3', PyL_2CR_estanco.columns[2:]] = c3_sum.values
PyL_2CR_estanco.loc[PyL_2CR_estanco['LOCAL'] == 'c4', PyL_2CR_estanco.columns[2:]] = c4_sum.values
PyL_2CR_estanco.loc[PyL_2CR_estanco['LOCAL'] == 'c5', PyL_2CR_estanco.columns[2:]] = c5_sum.values
PyL_2CR_estanco.loc[PyL_2CR_estanco['LOCAL'] == 'c6', PyL_2CR_estanco.columns[2:]] = c6_sum.values
PyL_2CR_estanco.loc[PyL_2CR_estanco['LOCAL'] == 'c7', PyL_2CR_estanco.columns[2:]] = c7_sum.values
PyL_2CR_estanco.loc[PyL_2CR_estanco['LOCAL'] == 'c8', PyL_2CR_estanco.columns[2:]] = c8_sum.values
PyL_2CR_estanco.loc[PyL_2CR_estanco['LOCAL'] == 'c9', PyL_2CR_estanco.columns[2:]] = c9_sum.values
PyL_2CR_estanco.loc[PyL_2CR_estanco['LOCAL'] == 'c10', PyL_2CR_estanco.columns[2:]] = c10_sum.values
PyL_2CR_estanco.loc[PyL_2CR_estanco['LOCAL'] == 'c11', PyL_2CR_estanco.columns[2:]] = c11_sum.values
PyL_2CR_estanco.loc[PyL_2CR_estanco['LOCAL'] == 'c12', PyL_2CR_estanco.columns[2:]] = c12_sum.values
PyL_2CR_estanco.loc[PyL_2CR_estanco['LOCAL'] == 'c13', PyL_2CR_estanco.columns[2:]] = c13_sum.values
PyL_2CR_estanco.loc[PyL_2CR_estanco['LOCAL'] == 'c14', PyL_2CR_estanco.columns[2:]] = c14_sum.values
PyL_2CR_estanco.loc[PyL_2CR_estanco['LOCAL'] == 'c15', PyL_2CR_estanco.columns[2:]] = c15_sum.values
PyL_2CR_estanco.loc[PyL_2CR_estanco['LOCAL'] == 'c16', PyL_2CR_estanco.columns[2:]] = c16_sum.values
PyL_2CR_estanco.loc[PyL_2CR_estanco['LOCAL'] == 'c17', PyL_2CR_estanco.columns[2:]] = c17_sum.values


# Supongamos que 'CONCEPTOS' es el nombre de la columna a partir de la cual deseas dividir las siguientes columnas
columna_dv_index = PyL_2CR_estanco.columns.get_loc('CONCEPTOS')
PyL_2CR_estanco.iloc[:, columna_dv_index+1:] = PyL_2CR_estanco.iloc[:, columna_dv_index+1:] / 1000000

<ipython-input-6-972c95376c7d>:20: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  PyL_2CR_estanco = PyL_2CR_acumulado_agrupado.groupby('CM').apply(calculate_differences)


#7) Cargue bases a Google Sheets

In [ ]:
# Actualizar hojas de cálculo en Google Sheets
output_sheet_balance_resultado = gc.open(input_workbook_name).worksheet('Balance_Resultado')
output_sheet_balance_resultado.clear()
output_sheet_balance_resultado.update([merged_df_concat_new.columns.values.tolist()] + merged_df_concat_new.fillna('').values.tolist())

output_sheet_balance_resultado = gc.open(input_workbook_name).worksheet('Nuevas_Cuentas')
existing_data = output_sheet_balance_resultado.get_all_values()
if 'fecha_actualizar' not in merged_df44.columns:
    merged_df44['fecha_actualizar'] = fecha_actualizar
header = merged_df44.columns.tolist()
new_data = merged_df44.fillna('').values.tolist()
if existing_data:
    combined_data = existing_data[1:] + new_data
else:
    combined_data = new_data
output_sheet_balance_resultado.update([header] + combined_data)

sheet6 = spreadsheet.worksheet("Nuevas_Cuentas")
data6_range = sheet6.get('A:F')
df66 = pd.DataFrame(data6_range[1:], columns=data6_range[0])
df66['cuenta'] = df66['cuenta'].str.replace(',', '').astype(int)
df66['valor'] = df66['valor'].str.replace(',', '').astype(float)
df66['validacion'] = df66['cuenta'].isin(merged_df_concat_new['cuenta'])
df66 = df66.sort_values(by='fecha_actualizar')
df66 = df66.drop_duplicates(subset='cuenta', keep='first')
df66 = df66[df66['Parameteria'] != 'TRUE']

output_sheet_balance_resultado = gc.open(input_workbook_name).worksheet('Nuevas_Cuentas')
output_sheet_balance_resultado.clear()
output_sheet_balance_resultado.update([df66.columns.values.tolist()] + df66.fillna('').values.tolist())

output_sheet_balance_resultado = gc.open(input_workbook_name).worksheet('Parametria_1')
output_sheet_balance_resultado.clear()
output_sheet_balance_resultado.update([df2.columns.values.tolist()] + df2.fillna('').values.tolist())


df22 = df22.rename(columns={'LOCAL': 'cuenta'})
output_sheet_balance_resultado = gc.open(input_workbook_name).worksheet('Parametria_2')
output_sheet_balance_resultado.clear()
output_sheet_balance_resultado.update([df22.columns.values.tolist()] + df22.fillna('').values.tolist())

output_sheet_balance_resultado = gc.open(input_workbook_name).worksheet('P&L_2CR')
output_sheet_balance_resultado.clear()
output_sheet_balance_resultado.update([PyL_2CR_estanco.columns.values.tolist()] + PyL_2CR_estanco.fillna('').values.tolist())

{'spreadsheetId': '1t1EDsT9HQQkYL5QnsmV0u2wDagduXrFm4OHqjqak5-8',
 'updatedRange': "'P&L_2CR'!A1:AY36",
 'updatedRows': 36,
 'updatedColumns': 51,
 'updatedCells': 1836}